In [2]:
# Transformer 모델, 문장 임베딩, 유사도 측정을 위한 라이브러리 설치
!pip install transformers sentence-transformers torch scikit-learn

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Two lists of sentences
sentences1 = [
    "The new movie is awesome",
    "The cat sits outside",
    "A man is playing guitar",
]

sentences2 = [
    "The dog plays in the garden",
    "The new movie is so great",
    "A woman watches TV",
]

# Compute embeddings for both lists
embeddings1 = model.encode(sentences1)
embeddings2 = model.encode(sentences2)

# Compute cosine similarities
similarities = model.similarity(embeddings1, embeddings2)

# Output the pairs with their score
for idx_i, sentence1 in enumerate(sentences1):
    print(sentence1)
    for idx_j, sentence2 in enumerate(sentences2):
        print(f" - {sentence2: <30}: {similarities[idx_i][idx_j]:.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The new movie is awesome
 - The dog plays in the garden   : 0.0543
 - The new movie is so great     : 0.8939
 - A woman watches TV            : -0.0502
The cat sits outside
 - The dog plays in the garden   : 0.2838
 - The new movie is so great     : -0.0029
 - A woman watches TV            : 0.1310
A man is playing guitar
 - The dog plays in the garden   : 0.2277
 - The new movie is so great     : -0.0136
 - A woman watches TV            : -0.0327


In [3]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


In [4]:
from sentence_transformers import CrossEncoder

# 1. Load a pretrained CrossEncoder model
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

# The texts for which to predict similarity scores
query = "How many people live in Berlin?"
passages = [
    "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers.",
    "Berlin has a yearly total of about 135 million day visitors, making it one of the most-visited cities in the European Union.",
    "In 2013 around 600,000 Berliners were registered in one of the more than 2,300 sport and fitness clubs.",
]

# 2a. predict scores for pairs of texts
scores = model.predict([(query, passage) for passage in passages])
print(scores)
# => [8.607139 5.506266 6.352977]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[8.6071415 5.5062675 6.352984 ]


In [5]:
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

# Load the BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Example sentences (already preprocessed)
tokens1 = ["[CLS]", "i", "like", "coding", "in", "python", ".", "[SEP]"]
tokens2 = ["[CLS]", "python", "is", "my", "favorite", "programming", "language", ".", "[SEP]"]

# Convert tokens to input IDs
input_ids1 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens1)).unsqueeze(0)  # Batch size 1
input_ids2 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens2)).unsqueeze(0)  # Batch size 1

# Obtain the BERT embeddings
with torch.no_grad():
    outputs1 = model(input_ids1)
    outputs2 = model(input_ids2)
    embeddings1 = outputs1.last_hidden_state[:, 0, :]  # [CLS] token
    embeddings2 = outputs2.last_hidden_state[:, 0, :]  # [CLS] token

# Calculate similarity
similarity_score = cosine_similarity(embeddings1, embeddings2)
print("Similarity Score:", similarity_score)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Similarity Score: [[0.9558883]]


In [6]:
from transformers import BertTokenizer

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Example sentences
sentence1 = "I like coding in Python."
sentence2 = "Python is my favorite programming language."

# Tokenize the sentences
tokens1 = tokenizer.tokenize(sentence1)
tokens2 = tokenizer.tokenize(sentence2)

# Add [CLS] and [SEP] tokens
tokens = ['[CLS]'] + tokens1 + ['[SEP]'] + tokens2 + ['[SEP]']
print("Token:", tokens)

# Convert tokens to input IDs
input_ids = tokenizer.convert_tokens_to_ids(tokens)

# Display the tokens and input IDs
print("Input IDs:", input_ids)

from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

# Load the BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Convert tokens to input IDs
input_ids1 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens1)).unsqueeze(0)  # Batch size 1
input_ids2 = torch.tensor(tokenizer.convert_tokens_to_ids(tokens2)).unsqueeze(0)  # Batch size 1

# Obtain the BERT embeddings
with torch.no_grad():
    outputs1 = model(input_ids1)
    outputs2 = model(input_ids2)
    embeddings1 = outputs1.last_hidden_state[:, 0, :]  # [CLS] token
    embeddings2 = outputs2.last_hidden_state[:, 0, :]  # [CLS] token

# Calculate similarity
similarity_score = cosine_similarity(embeddings1, embeddings2)
print("Similarity Score:", similarity_score)


Token: ['[CLS]', 'i', 'like', 'coding', 'in', 'python', '.', '[SEP]', 'python', 'is', 'my', 'favorite', 'programming', 'language', '.', '[SEP]']
Input IDs: [101, 1045, 2066, 16861, 1999, 18750, 1012, 102, 18750, 2003, 2026, 5440, 4730, 2653, 1012, 102]
Similarity Score: [[0.38574767]]


## Transformer 과제 수행

In [8]:
# 임포트
import torch
import numpy as np
import itertools
from sentence_transformers import SentenceTransformer
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
# 내가 사용할 캡션 정의
images_and_captions = {
    "2008_008231.jpg": [
        "(1) A beautiful view in the mountains is taken in by a man wearing shorts.",
        "(2) A man on the summit of a mountain looking at a glider in the sky.",
        "(3) A man stands on a mountain, watching a plane.",
        "(4) A man wearing shorts stands in the snow at the edge of a cliff.",
        "(5) Professor standing on a snowy mountain, watching a plane."
    ],
    "2008_005279.jpg": [
        "(1) A blue and green hummingbird hovers near a pink flower.",
        "(2) A blue hummingbird is flying by a flower.",
        "(3) A hummingbird in front of a flower",
        "(4) A small hummingbird flying near a pink flower.",
        "(5) Colorful hummingbird hovering in front of red flower."
    ],
    "2008_007247.jpg": [
        "(1) A black train engine is facing me on the tracks with its light on in the woods during the day.",
        "(2) A close-up of a black train engine.",
        "(3) A front view of a black train engine on the tracks with people standing on either side.",
        "(4) boy feeding sheep hay",
        "(5) The train is very old and has the number 1225 on it."
    ]
}

In [9]:
# 유사도 순위 정렬 함수 정의
def get_ranked_pairs(captions, similarity_matrix):
    # 유사도 행렬을 기반으로 모든 캡션 쌍의 유사도를 추출하고 높은 순서로 정렬
    similarity_list = []
    n = len(captions)

    # itertools.combinations를 사용하여 중복 없이 모든 쌍을 추출
    for i, j in itertools.combinations(range(n), 2):
        # 캡션 번호만 추출하여 쌍 이름 생성
        try:
            pair_name = f"({captions[i].split(')')[0]})/({captions[j].split(')')[0]})"
        except IndexError:
            # 캡션이 번호가 없을 경우
            pair_name = f"Pair {i+1}/{j+1}"

        score = similarity_matrix[i][j]
        similarity_list.append((score, pair_name))

    # 유사도 수치 순서로 정렬
    similarity_list.sort(key=lambda x: x[0], reverse=True)
    return similarity_list

### 1. Mini 모델

In [10]:
# Mini 모델 로드
mini_model = SentenceTransformer('all-MiniLM-L6-v2')

for image, captions in images_and_captions.items():
    print(f"\n--- 사진: {image} ---")

    # 임베딩 생성
    caption_embeddings = mini_model.encode(captions, convert_to_numpy=True)

    # 코사인 유사도 행렬 계산
    similarity_matrix = cosine_similarity(caption_embeddings)

    # 유사도 측정 및 순위 정렬
    ranked_results = get_ranked_pairs(captions, similarity_matrix)

    # 결과 출력
    for score, pair in ranked_results:
        print(f"유사도: {score:.4f} | 쌍: {pair}")


--- 사진: 2008_008231.jpg ---
유사도: 0.7128 | 쌍: ((3)/((5)
유사도: 0.6391 | 쌍: ((2)/((3)
유사도: 0.6117 | 쌍: ((1)/((2)
유사도: 0.6089 | 쌍: ((1)/((4)
유사도: 0.5952 | 쌍: ((1)/((3)
유사도: 0.5091 | 쌍: ((2)/((5)
유사도: 0.4582 | 쌍: ((4)/((5)
유사도: 0.4322 | 쌍: ((3)/((4)
유사도: 0.4199 | 쌍: ((2)/((4)
유사도: 0.4147 | 쌍: ((1)/((5)

--- 사진: 2008_005279.jpg ---
유사도: 0.8679 | 쌍: ((1)/((4)
유사도: 0.8629 | 쌍: ((1)/((2)
유사도: 0.8534 | 쌍: ((1)/((3)
유사도: 0.8489 | 쌍: ((1)/((5)
유사도: 0.8439 | 쌍: ((3)/((4)
유사도: 0.8397 | 쌍: ((2)/((4)
유사도: 0.8367 | 쌍: ((4)/((5)
유사도: 0.8298 | 쌍: ((2)/((3)
유사도: 0.8221 | 쌍: ((3)/((5)
유사도: 0.7612 | 쌍: ((2)/((5)

--- 사진: 2008_007247.jpg ---
유사도: 0.7071 | 쌍: ((1)/((2)
유사도: 0.6475 | 쌍: ((1)/((3)
유사도: 0.6467 | 쌍: ((2)/((3)
유사도: 0.4521 | 쌍: ((2)/((5)
유사도: 0.3920 | 쌍: ((3)/((5)
유사도: 0.3774 | 쌍: ((1)/((5)
유사도: 0.1472 | 쌍: ((2)/((4)
유사도: 0.1375 | 쌍: ((1)/((4)
유사도: 0.1195 | 쌍: ((4)/((5)
유사도: 0.0734 | 쌍: ((3)/((4)


### 2. BERT 모델

In [11]:
# BERT 모델 및 토크나이저 로드
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# GPU 사용 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)


def get_bert_embeddings_list(captions, tokenizer, model, device):
    #BERT 모델의 토큰 임베딩을 추출
    embeddings_list = []
    # 5개의 캡션을 하나씩 처리
    for caption in captions:
        # 토큰화 및 Input ID 변환
        encoded_input = tokenizer(caption, return_tensors='pt', padding=True, truncation=True).to(device)

        with torch.no_grad():
            output = model(**encoded_input)
            # 토큰 임베딩 추출
            cls_embedding = output.last_hidden_state[:, 0, :].squeeze()
            embeddings_list.append(cls_embedding.cpu().numpy())

    return np.array(embeddings_list) # NumPy 배열로 변환

In [13]:
for image, captions in images_and_captions.items():
    print(f"\n--- 사진: {image} ---")

    # 임베딩 생성
    bert_embeddings = get_bert_embeddings_list(captions, bert_tokenizer, bert_model, device)

    # 코사인 유사도 행렬 계산
    similarity_matrix = cosine_similarity(bert_embeddings)

    # 유사도 측정 및 순위 정렬
    ranked_results = get_ranked_pairs(captions, similarity_matrix)

    # 결과 출력
    for score, pair in ranked_results:
        print(f"유사도: {score:.4f} | 쌍: {pair}")


--- 사진: 2008_008231.jpg ---
유사도: 0.9521 | 쌍: ((3)/((4)
유사도: 0.9468 | 쌍: ((2)/((5)
유사도: 0.9267 | 쌍: ((2)/((3)
유사도: 0.9241 | 쌍: ((3)/((5)
유사도: 0.9021 | 쌍: ((1)/((2)
유사도: 0.9007 | 쌍: ((1)/((3)
유사도: 0.8954 | 쌍: ((1)/((4)
유사도: 0.8882 | 쌍: ((2)/((4)
유사도: 0.8858 | 쌍: ((1)/((5)
유사도: 0.8822 | 쌍: ((4)/((5)

--- 사진: 2008_005279.jpg ---
유사도: 0.9530 | 쌍: ((4)/((5)
유사도: 0.9500 | 쌍: ((2)/((4)
유사도: 0.9422 | 쌍: ((1)/((2)
유사도: 0.9363 | 쌍: ((2)/((3)
유사도: 0.9329 | 쌍: ((3)/((4)
유사도: 0.9238 | 쌍: ((3)/((5)
유사도: 0.9232 | 쌍: ((1)/((4)
유사도: 0.9190 | 쌍: ((2)/((5)
유사도: 0.9031 | 쌍: ((1)/((3)
유사도: 0.8973 | 쌍: ((1)/((5)

--- 사진: 2008_007247.jpg ---
유사도: 0.9486 | 쌍: ((2)/((3)
유사도: 0.8451 | 쌍: ((2)/((4)
유사도: 0.8446 | 쌍: ((1)/((3)
유사도: 0.8367 | 쌍: ((3)/((4)
유사도: 0.8144 | 쌍: ((1)/((2)
유사도: 0.7858 | 쌍: ((1)/((5)
유사도: 0.7685 | 쌍: ((2)/((5)
유사도: 0.7676 | 쌍: ((3)/((5)
유사도: 0.7586 | 쌍: ((1)/((4)
유사도: 0.7291 | 쌍: ((4)/((5)
